# Week 4 — Phase 1: 계정 마스터 생성 및 검증

**목표**: `master_gl_clean.csv` 에서 Prefix 기반 계정 마스터(`account_master.csv`)를 생성하고 검증

**순서**
1. master_gl_clean.csv 로드
2. `generate_master()` 실행 → `data/account_master.csv` 생성
3. 생성 결과 검증 (행 수 · 앞자리 0 · 계층 분포)
4. lv1_name / lv2_name 수동 편집 안내
5. `load_data()` 조인 검증
6. `get_account_list()` 동작 확인

## 0. 경로 설정

In [2]:
import sys, os
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analyzer import generate_master, load_data, get_account_list

GL_PATH     = os.path.join(PROJECT_ROOT, "data", "master_gl_clean.csv")
MASTER_PATH = os.path.join(PROJECT_ROOT, "data", "account_master.csv")
os.makedirs(os.path.dirname(MASTER_PATH), exist_ok=True)

print("GL 경로   :", GL_PATH)
print("마스터 경로:", MASTER_PATH)

GL 경로   : g:\내 드라이브\03_DArt-B\12_정규스터디\project\data\master_gl_clean.csv
마스터 경로: g:\내 드라이브\03_DArt-B\12_정규스터디\project\data\account_master.csv


## 1. master_gl_clean.csv 로드

In [3]:
gl = pd.read_csv(GL_PATH, dtype={"acc_code": str, "doc_no": str}, low_memory=False)
print(f"행 수: {len(gl):,}  |  컬럼 수: {len(gl.columns)}")
gl[["acc_code", "acc_name"]].drop_duplicates().head(10)

행 수: 332,103  |  컬럼 수: 34


,acc_code,acc_name
0,500010,Transporting
43,500020,Forwarding charges
47,510020,Consumption - Finished Goods
67,191100,GR/IR-clearing - external procurement
81,231500,Loss from price differences of own products
97,300000,Raw material 1
100,792000,Finished goods
130,281500,Gain from own product price differences
145,121000,Unfinished products
12119,140000,Customers - Domestic Receivables


## 2. generate_master() 실행

In [4]:
master = generate_master(gl, MASTER_PATH)
print(f"생성된 마스터 행 수: {len(master)}  (unique acc_code 개수와 일치해야 함)")
master

생성된 마스터 행 수: 63  (unique acc_code 개수와 일치해야 함)


,acc_code,acc_code_clean,acc_name,lv1_code,lv1_name,lv2_code,lv2_name
0,113000,113000,Land,1,(임시)1,113,(임시)113
1,113006,113006,Concentration Account - Checks In,1,(임시)1,113,(임시)113
2,113016,113016,Lockbox Account - Checks In,1,(임시)1,113,(임시)113
3,113101,113101,Bank 1 (checks payable),1,(임시)1,113,(임시)113
4,121000,121000,Unfinished products,1,(임시)1,121,(임시)121
...,...,...,...,...,...,...,...
58,792000,792000,Finished goods,7,(임시)7,792,(임시)792
59,799999,799999,Initial entry of stock balances(own/trading go...,7,(임시)7,799,(임시)799
60,800000,800000,Sales revenues - domestic,8,(임시)8,800,(임시)800
61,889000,889000,Other sales deductions,8,(임시)8,889,(임시)889


## 3. 검증

In [5]:
n_unique = gl["acc_code"].nunique()
assert len(master) == n_unique, f"행 수 불일치: master={len(master)}, unique acc_code={n_unique}"
print(f"[OK] 행 수 일치: {len(master)}")

leading_zero = master[
    master["acc_code_clean"].str.startswith("0") & (master["acc_code_clean"] != "0")
]
if len(leading_zero) == 0:
    print("[OK] acc_code_clean 앞자리 0 없음")
else:
    print(f"[WARN] 앞자리 0 남아있는 항목 {len(leading_zero)}개:")
    display(leading_zero)

print("\n[lv1_code 분포]")
print(master["lv1_code"].value_counts().to_string())
print("\n[lv2_code 분포]")
print(master["lv2_code"].value_counts().to_string())

[OK] 행 수 일치: 63
[OK] acc_code_clean 앞자리 0 없음

[lv1_code 분포]
lv1_code
2    19
1    13
4     8
6     7
5     6
7     4
3     3
8     3

[lv2_code 분포]
lv2_code
216    6
113    4
500    3
510    3
410    3
399    2
217    2
231    2
610    2
440    2
700    2
630    2
191    1
154    1
141    1
140    1
134    1
125    1
121    1
175    1
160    1
232    1
235    1
230    1
204    1
211    1
400    1
300    1
282    1
294    1
281    1
260    1
449    1
417    1
620    1
650    1
660    1
792    1
799    1
800    1
889    1
893    1


## 4. lv1_name / lv2_name 수동 편집 안내

생성된 `data/account_master.csv` 의 `lv1_name`, `lv2_name` 컬럼은 `(임시){code}` 형태입니다.  
**엑셀 또는 CSV 에디터**로 열어 실제 분류명으로 수정 후 저장하세요.

| lv1_code | 수정 전 | 수정 후 예시 |
|----------|---------|--------------|
| 1 | (임시)1 | 자산 |
| 2 | (임시)2 | 부채 |
| 4 | (임시)4 | 수익 |
| 5 | (임시)5 | 비용 |

In [6]:
master_edited = pd.read_csv(MASTER_PATH, dtype=str)
still_tmp = master_edited[
    master_edited["lv1_name"].str.startswith("(임시)") |
    master_edited["lv2_name"].str.startswith("(임시)")
]
if len(still_tmp) > 0:
    print(f"[INFO] 편집 필요 항목: {len(still_tmp)}개")
    display(still_tmp[["lv1_code", "lv1_name", "lv2_code", "lv2_name"]].drop_duplicates())
else:
    print("[OK] 모든 분류명 편집 완료")

[OK] 모든 분류명 편집 완료


## 5. load_data() 조인 검증

In [7]:
df = load_data(GL_PATH, MASTER_PATH)
print(f"조인 결과 행 수: {len(df):,}  (GL 원본 {len(gl):,}와 같아야 함)")
assert len(df) == len(gl), "행 수 변화 — acc_code 누락 확인 필요"

null_lv1 = df["lv1_code"].isna().sum()
if null_lv1 > 0:
    print(f"[WARN] lv1_code 누락 {null_lv1}건")
    display(df[df["lv1_code"].isna()][["acc_code", "acc_name"]].drop_duplicates())
else:
    print("[OK] 모든 행에 계층 정보 조인됨")

df.head()

조인 결과 행 수: 332,103  (GL 원본 332,103와 같아야 함)
[OK] 모든 행에 계층 정보 조인됨


,doc_no,company_code,fisc_year,line_no,post_date,doc_date,fisc_month,acc_code,acc_name,acc_type,...,_is_reversal_flag,_flag_weekend,_flag_reversal,_flag_out_of_period,_flag_duplicate,acc_code_clean,lv1_code,lv1_name,lv2_code,lv2_name
0,5105600151,C003,2022,2,2022-04-24,2022-04-24,04월,500010,Transporting,손익계산서(PL),...,False,True,False,False,False,500010,5,매출,500,매출액
1,5105600143,C003,2022,2,2022-04-24,2022-04-24,04월,500010,Transporting,손익계산서(PL),...,False,True,False,False,False,500010,5,매출,500,매출액
2,5105600152,C003,2022,2,2022-04-24,2022-04-24,04월,500010,Transporting,손익계산서(PL),...,False,True,False,False,False,500010,5,매출,500,매출액
3,5105600153,C003,2022,2,2022-04-25,2022-04-25,04월,500010,Transporting,손익계산서(PL),...,False,False,False,False,False,500010,5,매출,500,매출액
4,5105600154,C003,2022,2,2022-04-25,2022-04-25,04월,500010,Transporting,손익계산서(PL),...,False,False,False,False,False,500010,5,매출,500,매출액


## 6. get_account_list() 동작 확인

In [8]:
master_df = pd.read_csv(MASTER_PATH, dtype=str)

lv1_list = get_account_list(master_df, level=1)
print("[대분류 목록]")
for item in lv1_list:
    print(f"  {item['code']} — {item['name']}")

if lv1_list:
    first_lv1 = lv1_list[0]["code"]
    lv2_list = get_account_list(master_df, level=2, parent=first_lv1)
    print(f"\n[중분류 — 대분류 '{first_lv1}']")
    for item in lv2_list:
        print(f"  {item['code']} — {item['name']}")

    if lv2_list:
        first_lv2 = lv2_list[0]["code"]
        lv3_list = get_account_list(master_df, level=3, parent=first_lv2)
        print(f"\n[소분류 — 중분류 '{first_lv2}']")
        for item in lv3_list:
            print(f"  {item['code']} — {item['name']}")

[대분류 목록]
  1 — 자산
  2 — 부채 및 조정계정
  3 — 재고자산
  4 — 매출원가 및 매입채무
  5 — 매출
  6 — 판매비와관리비
  7 — 기타손익
  8 — 매출 및 재고변동

[중분류 — 대분류 '1']
  113 — 현금 및 예금
  121 — 재공품
  125 — 미수수익
  134 — 특수관계자 채권
  140 — 국내 매출채권
  141 — 해외 매출채권
  154 — 매입부가세
  160 — 국내 매입채무
  175 — 매출부가세
  191 — GR/IR 청산계정

[소분류 — 중분류 '113']
  113000 — 토지
  113006 — 수표 집중계좌
  113016 — 락박스 수표 입금계좌
  113101 — 은행 1 (지급수표)
